# Add your own city to CitiesTab

Your original spectral data may come from any city, instrument, sampling schedule, or storage format. It only needs a time for each measured spectrum, wavelengths in nm, and spectral values. This standalone example reads a configurable long CSV and converts it into the strict wide monthly average-day CSV used by `CitiesTab.tsx`.

**Granada is only the included example.** Replace the input path, city metadata, and column mapping in the next cell for your own data.

In [1]:
from pathlib import Path
import calendar
import pandas as pd

# ---- Change this configuration for your city ----
INPUT_CSV = Path("example_input/ES-UGR_Granada_spectral_horizontal_irradiance.csv")
OUTPUT_CSV = Path("output/ES-UGR_Granada_average_day_by_month.csv")

LOCATION_CODE = "ES-UGR"
LOCATION_NAME = "Granada"
MEASUREMENT_TABLE = "spectral_horizontal_irradiance"
BIN_MINUTES = 5

# Map standard fields to your input columns. Use None when a field is absent.
COLUMNS = {
    "location_code": "location_code",
    "timestamp": "timestamp",
    "wavelength": "wavelength",
    "value": "spectral_horizontal_irradiance",
    "measurement_setup": "measurement_setup",
    "sun_included": "sun_included",
    "patch": None,
    "patch_almucantar": None,
    "patch_azimuth": None,
}

DEFAULTS = {
    "measurement_setup": "1",
    "sun_included": "TRUE",
    "patch": "",
    "patch_almucantar": "",
    "patch_azimuth": "",
}

## 1. Load and validate

The city pipeline uses local civil time. A trailing timezone abbreviation is accepted but is not converted to UTC.

In [2]:
if BIN_MINUTES <= 0:
    raise ValueError("BIN_MINUTES must be greater than zero")

raw = pd.read_csv(INPUT_CSV, low_memory=False)
required_fields = ("timestamp", "wavelength", "value")
missing = [
    COLUMNS[field] for field in required_fields
    if not COLUMNS.get(field) or COLUMNS[field] not in raw.columns
]
if missing:
    raise ValueError(f"Missing required input columns: {missing}")

location_column = COLUMNS.get("location_code")
if location_column:
    if location_column not in raw.columns:
        raise ValueError(f"Missing location column: {location_column}")
    raw = raw.loc[raw[location_column].astype("string").str.strip() == LOCATION_CODE].copy()
if raw.empty:
    raise ValueError(f"No input rows remain for {LOCATION_CODE}")

data = pd.DataFrame({
    "timestamp_text": raw[COLUMNS["timestamp"]].astype("string").str.strip(),
    "wavelength": pd.to_numeric(raw[COLUMNS["wavelength"]], errors="coerce"),
    "value": pd.to_numeric(raw[COLUMNS["value"]], errors="coerce"),
})
for field in DEFAULTS:
    source_column = COLUMNS.get(field)
    if source_column and source_column in raw.columns:
        data[field] = raw[source_column].astype("string").fillna("").str.strip()
    else:
        data[field] = DEFAULTS[field]

data["sun_included"] = data["sun_included"].str.upper().replace({"1": "TRUE", "0": "FALSE"})
datetime_text = data["timestamp_text"].str.replace(
    r"\s+(?:[A-Za-z]+|[+-]\d{2}(?::?\d{2})?)$", "", regex=True
)
data["datetime"] = pd.to_datetime(datetime_text, errors="coerce")

if data["datetime"].isna().any():
    examples = data.loc[data["datetime"].isna(), "timestamp_text"].head(3).tolist()
    raise ValueError(f"Could not parse timestamp values such as: {examples}")
if data[["wavelength", "value"]].isna().any().any():
    raise ValueError("Wavelength and spectral value columns must be numeric and non-empty")
if not data["wavelength"].map(lambda value: float(value) == value and abs(value) != float("inf")).all():
    raise ValueError("Wavelengths must be finite numbers")
if not data["value"].map(lambda value: abs(value) != float("inf")).all():
    raise ValueError("Spectral values must be finite numbers")

print(f"Loaded {len(data):,} spectral rows for {LOCATION_NAME}.")

Loaded 108,054 spectral rows for Granada.


## 2. Average by month and 5-minute time bin

Separate measurement setups and optional patch dimensions remain separate slices, matching `Cities.py`.

In [3]:
dimensions = [
    "measurement_setup", "sun_included", "patch",
    "patch_almucantar", "patch_azimuth",
]
data["month"] = data["datetime"].dt.month.astype(int)
seconds = data["datetime"].dt.hour * 3600 + data["datetime"].dt.minute * 60 + data["datetime"].dt.second
data["time_bin_minutes"] = ((seconds // (BIN_MINUTES * 60)) * BIN_MINUTES).astype(int)

group_keys = ["month", "time_bin_minutes", *dimensions, "wavelength"]
averaged = data.groupby(group_keys, dropna=False, sort=False)["value"].agg(
    average_spectral_value="mean",
    samples_averaged="count",
).reset_index()

base_keys = ["month", "time_bin_minutes", *dimensions]
sample_counts = averaged.groupby(base_keys, dropna=False, sort=False)["samples_averaged"].min().reset_index()
wide = averaged.pivot_table(
    index=base_keys,
    columns="wavelength",
    values="average_spectral_value",
    aggfunc="first",
).reset_index()
wide.columns.name = None
wide = wide.merge(sample_counts, on=base_keys, how="left")

## 3. Format and export the CitiesTab CSV

In [4]:
def wavelength_label(value):
    number = float(value)
    return str(int(number)) if number.is_integer() else f"{number:g}"

wavelength_columns = sorted(
    [column for column in wide.columns if isinstance(column, (int, float))],
    key=float,
)
wide = wide.rename(columns={column: wavelength_label(column) for column in wavelength_columns})
wavelength_labels = [wavelength_label(column) for column in wavelength_columns]

wide.insert(0, "location_code", LOCATION_CODE)
wide.insert(1, "location_name", LOCATION_NAME)
wide.insert(2, "measurement_table", MEASUREMENT_TABLE)
wide.insert(4, "month_name", wide["month"].map(lambda month: calendar.month_name[int(month)]))
wide.insert(6, "time_of_day", wide["time_bin_minutes"].map(
    lambda minute: f"{int(minute) // 60:02d}:{int(minute) % 60:02d}"
))

metadata_columns = [
    "location_code", "location_name", "measurement_table", "month", "month_name",
    "time_bin_minutes", "time_of_day", *dimensions, "samples_averaged",
]
output = wide[metadata_columns + wavelength_labels].sort_values(
    ["measurement_table", "month", "time_bin_minutes", *dimensions]
).reset_index(drop=True)

slice_keys = ["measurement_table", "month", "time_bin_minutes", *dimensions]
if output.duplicated(slice_keys).any():
    raise ValueError("Output contains duplicate measurement slices for a time bin")
if output[wavelength_labels].isna().any().any():
    raise ValueError("At least one output row is missing a wavelength value")

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
output.to_csv(OUTPUT_CSV, index=False)
print(f"Created: {OUTPUT_CSV}")
print(f"Rows: {len(output):,} | Months: {output['month'].nunique()} | Wavelengths: {len(wavelength_labels)}")
output.head()

Created: output\ES-UGR_Granada_average_day_by_month.csv
Rows: 753 | Months: 12 | Wavelengths: 81


,location_code,location_name,measurement_table,month,month_name,time_bin_minutes,time_of_day,measurement_setup,sun_included,patch,...,735,740,745,750,755,760,765,770,775,780
0,ES-UGR,Granada,spectral_horizontal_irradiance,1,January,595,09:55,1,TRUE,,...,0.095265,0.097697,0.099648,0.099729,0.096376,0.059181,0.051173,0.087462,0.094471,0.094063
1,ES-UGR,Granada,spectral_horizontal_irradiance,1,January,605,10:05,1,TRUE,,...,0.094449,0.098031,0.100368,0.099927,0.095809,0.056026,0.050701,0.088051,0.094966,0.094947
2,ES-UGR,Granada,spectral_horizontal_irradiance,1,January,615,10:15,1,TRUE,,...,0.088914,0.093394,0.096099,0.095918,0.091952,0.054351,0.049717,0.084497,0.091140,0.091140
3,ES-UGR,Granada,spectral_horizontal_irradiance,1,January,620,10:20,1,TRUE,,...,0.093494,0.096843,0.099084,0.098902,0.095203,0.058100,0.052826,0.087704,0.094097,0.093889
4,ES-UGR,Granada,spectral_horizontal_irradiance,1,January,630,10:30,1,TRUE,,...,0.096173,0.098870,0.100998,0.101207,0.098735,0.059539,0.050080,0.088127,0.095908,0.095550


## Output format recap

The output contains exactly one city. Each row is one complete spectrum for one month and local time bin. `location_name`, `month`, and `time_bin_minutes` are mandatory. Every numeric column header after the metadata is interpreted as a wavelength in nm. Rows must be unique within their measurement setup and optional patch dimensions.

This notebook writes energy values in `W/(m2*nm)` without a `unit` column, so CitiesTab converts them to photon units. If your values are already `umol/(s*m2*nm)`, add a `unit` column containing `photon` on every output row.

Upload the generated file under **Cities > Source: Cities > City average-day CSV**, then upload a room calibration CSV and select a month.

https://github.com/NPEC-NL/Faketotron/blob/NPEC-WU-Dev/V2/workspace/web/src/ui/CitiesTab.tsx

Above is the typescript code of the Cities tab, so if uploading gives errors, there you can inspect the code.